[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_answering.ipynb)

# DataSage Stage 3: Answering GRPO Training

Pre-builds dataset with real environment observations (no `rollout_func`).
Reward functions replay the env with stored seeds for context-matched evaluation.
Includes Patronus Lynx integration for hallucination detection (optional).

**Fully self-contained** — runs in Colab with no repo clone or project imports.

**Requirements:** GPU (A100/H100), `HF_TOKEN`, `WANDB_API_KEY`.  
**Optional:** `PATRONUS_API_KEY` for Patronus Lynx.

In [ ]:
# Setup: install dependencies
!pip install -q unsloth trl datasets wandb requests
!pip install -q patronus  # optional

In [ ]:
# Load API keys from Colab Secrets or set manually
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    # Optional: for Patronus Lynx hallucination detection
    os.environ["PATRONUS_API_KEY"] = userdata.get("PATRONUS_API_KEY")
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."
# os.environ["PATRONUS_API_KEY"] = "pat_..."  # optional

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print("Keys loaded.")

In [ ]:
# Config + parser + reward helpers (all inlined, no project imports)
import json
import re
import requests
from dataclasses import dataclass, field

# ---- Config constants ----
WANDB_PROJECT = "datasage"
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
ENV_URL = "https://ricalanis-datasage-answering.hf.space"
HF_REPO = "ricalanis/datasage-qwen-answering"

LEARNING_RATE = 3e-6
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
NUM_GENERATIONS = 4
MAX_COMPLETION_LENGTH = 256
MAX_PROMPT_LENGTH = 512


# ---- Completion text extraction ----
def _get_text(completion) -> str:
    """Extract text from a completion (str or chat message list)."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        # Chat format: [{"role": "assistant", "content": "..."}]
        return completion[-1]["content"] if completion else ""
    return str(completion)


# ---- Parser ----
def parse_answering_action(text: str) -> dict:
    """Parse model output into an answering action dict."""
    json_match = re.search(r'\{[^{}]*"answer"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "answer" in data:
                return data
        except json.JSONDecodeError:
            pass

    cited = re.findall(r'\b([A-Z][a-zA-Z]+(?:[A-Z][a-z]+)*)\b', text)
    return {
        "answer": text,
        "cited_columns": cited[:5],
        "reasoning": "",
    }


# ---- Persona definitions (inlined from environments/shared/personas.py) ----
@dataclass
class Persona:
    name: str
    role: str
    focus_areas: list[str]
    language_style: str
    keywords: list[str]
    anti_keywords: list[str]


PERSONAS = [
    Persona(
        name="Executive", role="executive",
        focus_areas=["costs", "ROI", "strategic risk", "portfolio trends", "year-over-year"],
        language_style="strategic-financial",
        keywords=["revenue", "cost", "ROI", "risk", "trend", "quarter",
                  "year-over-year", "impact", "budget", "margin", "growth"],
        anti_keywords=["I think", "maybe", "um", "idk"],
    ),
    Persona(
        name="Manager", role="manager",
        focus_areas=["team performance", "operational health", "process bottlenecks", "capacity"],
        language_style="operational-actionable",
        keywords=["team", "performance", "bottleneck", "capacity", "SLA",
                  "process", "action", "priority", "escalation", "delivery"],
        anti_keywords=["shareholder", "valuation", "IPO"],
    ),
    Persona(
        name="Individual Contributor", role="ic",
        focus_areas=["personal tasks", "deadlines", "what to do next", "simple explanations"],
        language_style="plain-personal",
        keywords=["my", "I should", "next step", "deadline", "help",
                  "understand", "priority", "task", "assigned"],
        anti_keywords=["KPI", "ROI", "portfolio", "strategic", "EBITDA"],
    ),
]


def _check_formality(text: str, style: str) -> float:
    """Check if text formality matches the expected language style."""
    text_lower = text.lower()
    if style == "strategic-financial":
        indicators = ["percent", "%", "million", "billion", "quarter", "fiscal",
                      "forecast", "benchmark", "metric"]
        hits = sum(1 for ind in indicators if ind in text_lower)
        return min(hits / 2.0, 1.0)
    elif style == "operational-actionable":
        indicators = ["action", "recommend", "should", "priority", "next steps",
                      "immediate", "plan", "schedule"]
        hits = sum(1 for ind in indicators if ind in text_lower)
        return min(hits / 2.0, 1.0)
    elif style == "plain-personal":
        sentences = text.split(".")
        avg_len = sum(len(s.split()) for s in sentences) / max(len(sentences), 1)
        return 1.0 if avg_len < 20 else max(0.0, 1.0 - (avg_len - 20) / 30)
    return 0.5


def score_persona_alignment(answer: str, persona: Persona) -> float:
    """Score how well an answer aligns with a persona's communication style."""
    answer_lower = answer.lower()
    words = re.findall(r'\w+', answer_lower)
    word_count = max(len(words), 1)

    keyword_hits = sum(1 for kw in persona.keywords if kw.lower() in answer_lower)
    keyword_score = min(keyword_hits / max(len(persona.keywords) * 0.3, 1), 1.0)

    anti_hits = sum(1 for akw in persona.anti_keywords if akw.lower() in answer_lower)
    anti_penalty = min(anti_hits * 0.15, 0.5)

    formality_score = _check_formality(answer, persona.language_style)

    raw_score = 0.50 * keyword_score + 0.20 * formality_score + 0.30 - anti_penalty
    return round(max(0.0, min(1.0, raw_score)), 4)


# ---- Reward helpers ----
_ANSWER_JSON_RE = re.compile(r'\{[^{}]*"answer"[^{}]*\}', re.DOTALL)


def persona_match_reward(completions, **kwargs) -> list[float]:
    """Reward alignment with the REQUESTED persona (not just any persona)."""
    persona_names = kwargs.get("persona_name", [])
    if not persona_names:
        return [0.0] * len(completions)
    persona_map = {p.name: p for p in PERSONAS}
    rewards = []
    for completion, p_name in zip(completions, persona_names):
        text = _get_text(completion)
        persona = persona_map.get(p_name)
        if persona:
            rewards.append(score_persona_alignment(text, persona))
        else:
            rewards.append(0.0)
    return rewards


def local_faithfulness_fn(completions, **kwargs) -> list[float]:
    """Local faithfulness heuristic: checks column citations and data grounding."""
    rewards = []
    for completion in completions:
        text = _get_text(completion)
        score = 0.0
        cited = re.findall(r'\b([A-Z][a-zA-Z]+(?:[A-Z][a-z]+)*)\b', text)
        if len(cited) >= 1:
            score += 0.3
        if len(cited) >= 3:
            score += 0.2
        if re.search(r'\d+\.?\d*%', text) or re.search(r'\b\d{2,}\b', text):
            score += 0.2
        if any(w in text.lower() for w in ["i believe", "probably", "might be",
                                            "i'm not sure", "i think maybe"]):
            score -= 0.2
        if len(text.strip()) < 50:
            score -= 0.3
        rewards.append(max(0.0, min(1.0, score)))
    return rewards


def patronus_reward_fn(completions, **kwargs) -> list[float]:
    """Use Patronus Lynx for hallucination detection, with local fallback."""
    api_key = os.environ.get("PATRONUS_API_KEY")
    if not api_key:
        return local_faithfulness_fn(completions, **kwargs)
    try:
        import patronus
        patronus.init()
        from patronus import Patronus, RemoteEvaluator
        client = Patronus()
        lynx = RemoteEvaluator("lynx", "patronus:hallucination")
        rewards = []
        for completion in completions:
            text = _get_text(completion)
            result = client.evaluate(
                evaluators=lynx,
                task_output=text,
                task_input="Answer the question based on the data.",
                task_context="",
            )
            rewards.append(float(result.results[0].score))
        return rewards
    except Exception:
        return local_faithfulness_fn(completions, **kwargs)


def answering_json_format_reward(completions, **kwargs) -> list[float]:
    """Reward well-formed JSON answer actions."""
    rewards = []
    for completion in completions:
        text = _get_text(completion)
        match = _ANSWER_JSON_RE.search(text)
        if match:
            try:
                data = json.loads(match.group())
                has_answer = bool(data.get("answer", "").strip())
                has_cited = bool(data.get("cited_columns"))
                has_reasoning = bool(data.get("reasoning", "").strip())
                if has_answer and has_cited and has_reasoning:
                    rewards.append(1.0)
                elif has_answer and has_cited:
                    rewards.append(0.7)
                elif has_answer:
                    rewards.append(0.4)
                else:
                    rewards.append(0.2)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


print(f"Environment URL: {ENV_URL}")
print(f"Base model: {BASE_MODEL}")

In [ ]:
# Model loading via Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=True,
    fast_inference=True, max_lora_rank=16, gpu_memory_utilization=0.6,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded: {BASE_MODEL}")

In [ ]:
# System prompt and task descriptions
SYSTEM_PROMPT = """\
You are a data-driven enterprise analyst. You answer questions about \
datasets across multiple domains (HR, Sales, Project Management, \
IT Operations) tailored to the audience persona.

Personas:
- Executive: Focus on costs, ROI, strategic risk, portfolio trends, year-over-year metrics.
- Manager: Focus on team performance, operational health, process bottlenecks, capacity.
- Individual Contributor: Focus on personal tasks, deadlines, what to do next.

Respond with a JSON answer:
{"answer": "<your answer>", "cited_columns": ["col1", "col2"], "reasoning": "<chain-of-thought>"}

Rules:
1. Ground every claim in the data.
2. Match your tone and vocabulary to the persona.
3. Be concise but thorough.
4. Never fabricate numbers."""

TASK_DESCRIPTIONS = [
    "Answer the question based on the data, tailored to the persona.",
    "Provide a data-grounded answer appropriate for this audience.",
    "Analyze the data and answer the question in the persona's style.",
    "Use the dataset to answer accurately, matching the persona's focus.",
    "Generate a faithful, persona-aligned answer citing real data.",
    "Answer using statistics from the data, in the right tone for this persona.",
    "Review the data summary and answer the question for this stakeholder.",
    "Craft a response grounded in the data that matches the persona's needs.",
]

In [ ]:
# Dataset: pre-built with real environment observations
import random
from datasets import Dataset

random.seed(42)
SEEDS = [42 + i for i in range(64)]
prompts = []
persona_names = []

print("Building dataset with real environment observations...")
for i, seed in enumerate(SEEDS):
    task_desc = random.choice(TASK_DESCRIPTIONS)
    resp = requests.post(f"{ENV_URL}/web/reset", json={"seed": seed})
    resp.raise_for_status()
    obs = resp.json()["observation"]

    prompts.append([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Domain: {obs['domain']}\nPersona: {obs['persona']}\n"
            f"Persona Focus: {obs['persona_description']}\n\n"
            f"Question: {obs['question']}\n\n"
            f"Dataset Summary:\n{obs['dataset_summary']}\n\n"
            f"Column Statistics:\n{obs['column_stats']}\n\n"
            f"Available Columns: {', '.join(obs['available_columns'])}\n\n"
            f"Task: {task_desc}"
        )},
    ])
    persona_names.append(obs["persona"])
    if (i + 1) % 16 == 0:
        print(f"  Built {i + 1}/64 examples")

dataset = Dataset.from_dict({
    "prompt": prompts,
    "seed": SEEDS,
    "persona_name": persona_names,
})
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Environment reward function (calls env directly with stored seeds)
def answering_env_reward(completions, **kwargs) -> list[float]:
    """Primary reward: replay env with stored seed, step with parsed action."""
    seeds = kwargs.get("seed", [0] * len(completions))
    rewards = []
    for completion, seed in zip(completions, seeds):
        try:
            text = _get_text(completion)
            action_dict = parse_answering_action(text)
            requests.post(f"{ENV_URL}/web/reset", json={"seed": int(seed)})
            resp = requests.post(f"{ENV_URL}/web/step", json={"action": action_dict})
            resp.raise_for_status()
            rewards.append(float(resp.json().get("reward", 0.0)))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards

In [ ]:
# GRPO training config
from trl import GRPOConfig

training_args = GRPOConfig(
    output_dir="./outputs/answering-grpo",
    use_vllm=True,
    learning_rate=LEARNING_RATE,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_grad_norm=0.1,
    logging_steps=1, save_steps=50,
    report_to="wandb", run_name="datasage-answering-grpo-v2",
)
os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

In [ ]:
# Create trainer and train
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model, processing_class=tokenizer, args=training_args,
    train_dataset=dataset,
    reward_funcs=[answering_env_reward, patronus_reward_fn, answering_json_format_reward, persona_match_reward],
)
print("Starting Stage 3 (Answering) GRPO training...")
trainer.train()

In [ ]:
# Save and push to Hub
output_dir = "./outputs/answering-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

print(f"Pushing to Hub: {HF_REPO}")
trainer.push_to_hub(HF_REPO)
print(f"Model pushed to https://huggingface.co/{HF_REPO}")